In [1]:
import glob
import json
import os
import openpyxl
from openpyxl.styles import Alignment, Border, Side, Font, PatternFill
import pandas as pd

# Define output filename
output_excel = "Tennis_Analytics_Data.xlsx"

# Find all uploaded JSON files
json_files = glob.glob("*.json")
print(f"Found JSON files: {json_files}\n")

dfs = {}

for file_path in json_files:
    filename = os.path.basename(file_path).lower()
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # 1. Competitions
        if "atp" in filename or "competitions" in filename:
            comp_list = (
                data.get("competitions", []) if isinstance(data, dict) else data
            )
            df = pd.json_normalize(comp_list)
            cols = [
                c
                for c in [
                    "id",
                    "name",
                    "type",
                    "gender",
                    "level",
                    "category.name",
                    "parent_id",
                ]
                if c in df.columns
            ]
            if cols:
                df = df[cols]
            dfs["Competitions"] = df
            print(f"Processed Competitions ({len(df)} rows)")

        # 2. Complexes & Venues
        elif "complex" in filename:
            complex_list = (
                data.get("complexes", []) if isinstance(data, dict) else data
            )
            # Normalizing complexes and expanding nested venue counts/details
            complexes_rows = []
            for c in complex_list:
                venues = c.get("venues", [])
                complexes_rows.append(
                    {
                        "Complex ID": c.get("id"),
                        "Complex Name": c.get("name"),
                        "Total Venues/Courts": len(venues),
                        "Primary City": (
                            venues[0].get("city_name") if venues else "N/A"
                        ),
                        "Primary Country": (
                            venues[0].get("country_name") if venues else "N/A"
                        ),
                    }
                )
            df = pd.DataFrame(complexes_rows)
            dfs["Complexes"] = df
            print(f"Processed Complexes ({len(df)} rows)")

        # 3. Doubles Rankings
        elif "rankings" in filename or "doubles" in filename:
            rankings_list = []
            if isinstance(data, dict) and "rankings" in data:
                for group in data.get("rankings", []):
                    group_name = group.get("name", "Doubles")
                    for row in group.get("competitor_rankings", []):
                        player = row.get("competitor", {})
                        rankings_list.append(
                            {
                                "Tour": group_name,
                                "Rank": row.get("rank"),
                                "Points": row.get("points"),
                                "Competitions Played": row.get(
                                    "competitions_played"
                                ),
                                "Competitor ID": player.get("id"),
                                "Competitor Name": player.get("name"),
                                "Country": player.get("country"),
                                "Country Code": player.get("country_code"),
                            }
                        )
                df = pd.DataFrame(rankings_list)
            else:
                df = pd.json_normalize(data)
            dfs["Doubles Rankings"] = df
            print(f"Processed Rankings ({len(df)} rows)")

    except Exception as e:
        print(f"Error reading {file_path}: {e}")

# Build Excel file using OpenPyXL with styling
wb = openpyxl.Workbook()
wb.remove(wb.active)  # Remove default blank sheet

# Color scheme styling
header_fill = PatternFill(
    start_color="1F497D", end_color="1F497D", fill_type="solid"
)
header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
data_font = Font(name="Calibri", size=10)
zebra_fill = PatternFill(
    start_color="F9FAFB", end_color="F9FAFB", fill_type="solid"
)
thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)

for sheet_name, df in dfs.items():
    ws = wb.create_sheet(title=sheet_name)
    ws.views.sheetView[0].showGridLines = True

    # Column titles formatting
    headers = [
        str(c).replace("_", " ").replace(".", " ").title() for c in df.columns
    ]
    ws.append(headers)

    # Apply Header Styles
    for col_num, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_num)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")

    # Add Rows
    for r_idx, row in enumerate(df.itertuples(index=False), 2):
        ws.append(list(row))
        for c_idx, val in enumerate(row, 1):
            cell = ws.cell(row=r_idx, column=c_idx)
            cell.font = data_font
            cell.border = thin_border

            # Alternating background colors
            if r_idx % 2 == 1:
                cell.fill = zebra_fill

            # Formatting numeric values
            if isinstance(val, (int, float)):
                cell.alignment = Alignment(
                    horizontal="right", vertical="center"
                )
                if "Points" in headers[c_idx - 1] or "Rank" in headers[c_idx - 1]:
                    cell.number_format = "#,##0"
            else:
                cell.alignment = Alignment(
                    horizontal="left", vertical="center"
                )

    # Freeze Header Row
    ws.freeze_panes = "A2"

    # Auto-adjust column width
    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        col_letter = openpyxl.utils.get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 4, 12)

wb.save(output_excel)
print(f"\nExcel workbook successfully created: '{output_excel}'")

Found JSON files: ['doubles_competitor_rankings.json', 'complexes.json', 'atp_competitions.json']

Processed Rankings (1000 rows)
Processed Complexes (769 rows)
Processed Competitions (225 rows)

Excel workbook successfully created: 'Tennis_Analytics_Data.xlsx'


In [2]:
from google.colab import files

files.download("Tennis_Analytics_Data.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>